In [3]:
import ast
import string

import pandas as pd

DATASET_PATH = "../../data/out/distillation/hendrycks_math_distilled_detailed_raw_deepseek_v4_flash_regenerate_incorrect_w_large.parquet"

df = pd.read_parquet(DATASET_PATH)
print(f"Path: {DATASET_PATH}")
print(f"Shape: {df.shape}")
df.head(2)

Path: ../../data/out/distillation/hendrycks_math_distilled_detailed_raw_deepseek_v4_flash_regenerate_incorrect_w_large.parquet
Shape: (7500, 7)


,question,answer,answer_w_steps,question_id,distill_ans_correct,distill_reasoning,distill_answer
0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",0,"For the piecewise function to be continuous, t...",0,True,We need answer solve. Need think. Need provide...,To ensure the piecewise function is continuous...
1,A rectangular band formation is a formation wi...,98,Let $x$ be the number of band members in each ...,1,True,We need solve problem. Need provide step-by-st...,Let the original rectangular formation have \(...


## Accuracy


In [4]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 0.9889 (7417/7500)


## Missing distilled reasoning traces


In [12]:
missing_nan = int(df["distill_answer"].isna().sum())
empty_mask = df["distill_answer"].fillna("").str.strip() == ""
missing_or_empty = int(empty_mask.sum())
print(f"NaN distill_reasoning:            {missing_nan}")
print(f"Missing or empty distill_reasoning: {missing_or_empty}")

NaN distill_reasoning:            0
Missing or empty distill_reasoning: 1


## Reasoning trace length distribution (chars)


In [13]:
existing_reasoning = df.loc[~empty_mask, "distill_answer"]
lengths = existing_reasoning.str.len()
print(lengths.describe())
print(f"P60: {lengths.quantile(0.6)}")
print(f"P70: {lengths.quantile(0.7)}")
print(f"P80: {lengths.quantile(0.8)}")
print(f"P85: {lengths.quantile(0.85)}")
print(f"P90: {lengths.quantile(0.9)}")
print(f"P92: {lengths.quantile(0.92)}")
print(f"P95: {lengths.quantile(0.95)}")
print(f"P99: {lengths.quantile(0.99)}")

count    7499.000000
mean      690.752900
std       366.719904
min        93.000000
25%       419.500000
50%       604.000000
75%       878.000000
max      2824.000000
Name: distill_answer, dtype: float64
P60: 697.8000000000002
P70: 813.0
P80: 959.4000000000005
P85: 1062.0
P90: 1200.0
P92: 1266.0
P95: 1418.0
P99: 1852.1399999999967


In [ ]:
# corrected_empty_mask = df["corrected_reasoning"].fillna("").str.strip() == ""
# existing_corrected = df.loc[~corrected_empty_mask, "corrected_reasoning"]
# lengths = existing_corrected.str.len()
# print(lengths.describe())
# print(f"P60: {lengths.quantile(0.6)}")
# print(f"P70: {lengths.quantile(0.7)}")
# print(f"P80: {lengths.quantile(0.8)}")
# print(f"P85: {lengths.quantile(0.85)}")
# print(f"P90: {lengths.quantile(0.9)}")
# print(f"P92: {lengths.quantile(0.92)}")
# print(f"P95: {lengths.quantile(0.95)}")
# print(f"P99: {lengths.quantile(0.99)}")

count    1051.000000
mean      185.167460
std       141.933762
min        10.000000
25%        88.000000
50%       149.000000
75%       248.000000
max      1082.000000
Name: corrected_reasoning, dtype: float64
P60: 179.0
P70: 225.0
P80: 271.0
P85: 305.5
P90: 361.0
P92: 393.0
P95: 441.0
P99: 687.0


## Reasoning trace length distribution (tokens)

Token counts per reasoning trace under each of the three model tokenizers we use: `llama3b` (`meta-llama/Llama-3.2-3B-Instruct`), `phi4mini` (`microsoft/Phi-4-mini-instruct`), and `qwen3b` (`Qwen/Qwen2.5-3B-Instruct`).

In [27]:
# from tqdm.auto import tqdm
# from transformers import AutoTokenizer

# MODEL_TOKENIZERS = {
#     "llama3b": "meta-llama/Llama-3.2-3B-Instruct",
#     "phi4mini": "microsoft/Phi-4-mini-instruct",
#     "qwen3b": "Qwen/Qwen2.5-3B-Instruct",
# }

# reasoning_texts = existing_reasoning.tolist()

# for name, model_id in MODEL_TOKENIZERS.items():
#     tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
#     token_lengths = pd.Series(
#         [len(tokenizer(text, add_special_tokens=False)["input_ids"]) for text in tqdm(reasoning_texts, desc=name)],
#         index=existing_reasoning.index,
#     )
#     print(f"=== {name} ({model_id}) ===")
#     print(token_lengths.describe())
#     print(f"P60: {token_lengths.quantile(0.6)}")
#     print(f"P70: {token_lengths.quantile(0.7)}")
#     print(f"P80: {token_lengths.quantile(0.8)}")
#     print(f"P85: {token_lengths.quantile(0.85)}")
#     print(f"P90: {token_lengths.quantile(0.9)}")
#     print(f"P92: {token_lengths.quantile(0.92)}")
#     print(f"P95: {token_lengths.quantile(0.95)}")
#     print(f"P99: {token_lengths.quantile(0.99)}")
#     print()

## Invalid answers

An answer is valid iff, after stripping/lowercasing, it is one of the option letters `a`, `b`, ... up to the row's number of options.


In [14]:
LETTERS = list(string.ascii_lowercase)


def n_options(opts):
    if isinstance(opts, str):
        try:
            return len(ast.literal_eval(opts))
        except Exception:
            return 0
    try:
        return len(opts)
    except TypeError:
        return 0


def is_valid_answer(ans, n_opts):
    if not isinstance(ans, str):
        return False
    return ans.strip().lower() in set(LETTERS[:n_opts])


# n_opts_series = df["options"].apply(n_options)
valid_mask = df["distill_ans_correct"]
n_invalid = int((~valid_mask).sum())
print(f"Invalid distill_answer count: {n_invalid} ({n_invalid / n:.2%})")

Invalid distill_answer count: 83 (1.11%)


### All invalid answers


In [15]:
invalid_answers = df.loc[~valid_mask, "distill_answer"]
for idx, ans in invalid_answers.items():
    print(f"--- row {idx} ---")
    print(repr(ans))

--- row 334 ---
'To find the intersection points, set the two equations equal to each other:\n\n\\[\n2x^2-10x-10=x^2-4x+6\n\\]\n\nSubtract \\(x^2-4x+6\\) from both sides:\n\n\\[\n2x^2-10x-10-x^2+4x-6=0\n\\]\n\nCombine like terms:\n\n\\[\nx^2-6x-16=0\n\\]\n\nFactor the quadratic:\n\n\\[\n(x-8)(x+2)=0\n\\]\n\nSo,\n\n\\[\nx=8 \\quad \\text{or} \\quad x=-2\n\\]\n\nNow find the corresponding \\(y\\)-coordinates using \\(y=x^2-4x+6\\).\n\nFor \\(x=-2\\):\n\n\\[\ny=(-2)^2-4(-2)+6=4+8+6=18\n\\]\n\nFor \\(x=8\\):\n\n\\[\ny=8^2-4(8)+6=64-32+6=38\n\\]\n\nThus the intersection points are \\((-2,18)\\) and \\((8,38)\\).\n\nAnswer: (-2,18);(8,38)'
--- row 409 ---
'For the quadratic equation  \n\\[\nkx^2+20x+k=0\n\\]\nto have rational solutions, its discriminant must be a perfect square.\n\nThe discriminant is\n\\[\n\\Delta = 20^2 - 4(k)(k) = 400 - 4k^2 = 4(100-k^2).\n\\]\nSince \\(4\\) is already a perfect square, we need\n\\[\n100-k^2\n\\]\nto be a perfect square.\n\nAlso, the discriminant must be 

### 10 shortest reasoning traces


In [16]:
shortest = existing_reasoning.str.len().nsmallest(30)
for idx, length in shortest.items():
    print(f"--- row {idx} (len={length}) ---")
    print(df.loc[idx, "distill_answer"])
    print()

--- row 378 (len=93) ---
First add \(139+27=166\).

Next add \(166+23=189\).

Finally add \(189+11=200\).

Answer: 200

--- row 546 (len=104) ---
Use the difference of squares:
\[
98 \times 102 = (100-2)(100+2)=100^2-2^2=10000-4=9996.
\]
Answer: 9996

--- row 1142 (len=112) ---
When multiplying powers with the same base, add the exponents:
\[
t^3\cdot t^4 = t^{3+4} = t^7.
\]
Answer: t^{7}

--- row 6652 (len=114) ---
Combine like terms by adding the coefficients of \(y\):

\(2 + 3 + 4 = 9\)

So,

\(2y + 3y + 4y = 9y\)

Answer: 9y

--- row 5961 (len=117) ---
To find the value that is \(0.3\) more than \(29.8\), add \(0.3\) to \(29.8\):

\[
29.8 + 0.3 = 30.1
\]

Answer: 30.1

--- row 37 (len=121) ---
Substitute \(x=4\) into the function:
\[
f(4)=4^2-4
\]
Compute the square:
\[
4^2=16
\]
So:
\[
f(4)=16-4=12
\]
Answer: 12

--- row 6713 (len=121) ---
Write \(2.5\) as \(2.50\) so both numbers have two decimal places.

Then subtract:
\[
2.50 - 0.32 = 2.18
\]

Answer: 2.18

--- row 195 (len=123

## Reset distill columns for invalid answers

Sets `distill_reasoning` and `distill_answer` to `""` and `distill_ans_correct` to `False` for rows where the answer is invalid, then writes back to `DATASET_PATH`.


In [17]:
invalid_idx = df.index[~valid_mask]
df.loc[invalid_idx, "distill_reasoning"] = ""
df.loc[invalid_idx, "distill_answer"] = ""
df.loc[invalid_idx, "distill_ans_correct"] = False

df.to_parquet(DATASET_PATH, index=False)
print(f"Reset {len(invalid_idx)} rows and wrote {DATASET_PATH}")

Reset 83 rows and wrote ../../data/out/distillation/hendrycks_math_distilled_detailed_raw_deepseek_v4_flash_regenerate_incorrect_w_large.parquet


## Reset reasoning traces shorter than 10 chars

Sets `distill_reasoning` to `""` for rows where the existing trace is shorter than 10 characters, then writes back to `DATASET_PATH`.

In [32]:
# short_mask = df["distill_reasoning"].fillna("").str.len() < 200
# short_idx = df.index[short_mask]
# df.loc[short_idx, "distill_reasoning"] = ""
# df.loc[short_idx, "distill_answer"] = ""
# df.loc[short_idx, "distill_ans_correct"] = False
# print(f"distill_reasoning rows under {200} chars: {len(short_idx)}")

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")

In [6]:
max_reasoning_chars = 8192
long_mask = df["distill_reasoning"].str.len() > max_reasoning_chars
print(f"distill_reasoning rows over {max_reasoning_chars} chars: {int(long_mask.sum())}")
df.loc[long_mask, "distill_reasoning"] = (
    df.loc[long_mask, "distill_reasoning"].str.slice(stop=max_reasoning_chars)
)
# df.loc[long_mask, "distill_reasoning"] = ""
# df.loc[long_mask, "distill_answer"] = ""
# df.loc[long_mask, "distill_ans_correct"] = False

distill_reasoning rows over 8192 chars: 326


In [18]:
missing_reasoning_mask = df["distill_reasoning"].fillna("").eq("")
print(f"Dropping {int(missing_reasoning_mask.sum())} rows with NA or empty distill_reasoning")
df = df.loc[~missing_reasoning_mask].copy()

Dropping 83 rows with NA or empty distill_reasoning


In [64]:
max_corrected_reasoning_chars = 1444
max_combined_reasoning_chars = 8192

# df["corrected_reasoning"] = df["corrected_reasoning"].str.slice(
#     stop=max_corrected_reasoning_chars
# )
has_corrected_reasoning = df["corrected_reasoning"].fillna("").str.strip().ne("")
df.loc[has_corrected_reasoning, "distill_reasoning"] = [
    (reasoning if isinstance(reasoning, str) else "")[:
        max_combined_reasoning_chars - len(corrected)
    ] + corrected
    for reasoning, corrected in zip(
        df.loc[has_corrected_reasoning, "distill_reasoning"],
        df.loc[has_corrected_reasoning, "corrected_reasoning"],
    )
]

In [19]:
df["distill_reasoning"] = df["distill_answer"]

In [22]:
from core.utils.cot_answers import extract_math_answer

df["distill_answer"] = [extract_math_answer(assistant_response, []) for assistant_response in df["distill_answer"]]


In [24]:
df.to_parquet(
    "../../data/out/distillation/hendrycks_math_distilled_detailed_final.parquet", index=False
)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")